# Démonstration client réel — Omega Data API (C12)

Interroge l'API pour de vrai (`requests`, pas `TestClient`) : les tests
automatisés (`tests/integration/test_api_routes.py`) mockent la couche
d'accès aux données pour rester rapides et sans dépendance à Postgres ;
ce notebook, à l'inverse, démontre le comportement réel de bout en bout
contre une instance vivante de l'API, connectée à la base de travail
peuplée (C8 → C10 → C11).

**Prérequis** : API lancée et base peuplée —

```bash
docker compose -f infra/docker/docker-compose.yml --env-file .env up -d --build
./scripts/init_staging_db.sh && ./scripts/load_historique.sh
alembic upgrade head
python3 -m datacore.ingestion.run_extraction
python3 -m datacore.processing.run_cleaning
python3 -m datacore.storage.staging.load_staging
```

Voir [`docs/architecture/api_omega_data.md`](../docs/architecture/api_omega_data.md)
pour la documentation complète des routes et du modèle d'accès.


In [1]:
import requests

BASE_URL = "http://127.0.0.1:8000"

ENGINEER_KEY = "omega-data-engineer-2026"
ANALYST_KEY = "omega-data-analyst-2026"
NORDDRIVE_KEY = "omega-norddrive-2026"

def call(path, key=None, **params):
    """Appelle l'API et affiche le code HTTP + un aperçu de la réponse."""
    headers = {"X-API-Key": key} if key else {}
    resp = requests.get(f"{BASE_URL}{path}", headers=headers, params=params, timeout=5)
    print(f"GET {path} (params={params}) -> {resp.status_code}")
    try:
        body = resp.json()
        print(body if not isinstance(body, list) else body[:5])
    except ValueError:
        print(resp.text)
    return resp


## 1. Santé du service (sans authentification)

In [2]:
_ = call("/health")

GET /health (params={}) -> 200
{'status': 'ok', 'service': 'Omega Data API'}


## 2. Authentification

In [3]:
print("--- sans cle API ---")
_ = call("/commandes-clients")

print()
print("--- cle invalide ---")
_ = call("/commandes-clients", key="cle-invalide")


--- sans cle API ---
GET /commandes-clients (params={}) -> 401
{'detail': 'Clé API manquante ou invalide (en-tête X-API-Key)'}

--- cle invalide ---
GET /commandes-clients (params={}) -> 401
{'detail': 'Clé API manquante ou invalide (en-tête X-API-Key)'}


## 3. Autorisation — le point critique

Un Data Engineer peut consulter n'importe quel client. Un référent
client, lui, est **silencieusement ramené à son propre périmètre** même
s'il demande explicitement un autre client — pas juste rejeté.


In [4]:
print("--- Data Engineer : commandes NordDrive ---")
_ = call("/commandes-clients", key=ENGINEER_KEY, client="NordDrive", limit=2)

print()
print("--- referent NordDrive demandant FreshMarket : ramene a NordDrive ---")
_ = call("/commandes-clients", key=NORDDRIVE_KEY, client="FreshMarket", limit=2)


--- Data Engineer : commandes NordDrive ---
GET /commandes-clients (params={'client': 'NordDrive', 'limit': 2}) -> 200
[{'id': 1, 'client': 'NordDrive', 'commande_id': 'ND-000549', 'date_commande': '2026-07-19', 'entrepot': 'OMG-LIL'}, {'id': 2, 'client': 'NordDrive', 'commande_id': 'ND-000220', 'date_commande': '2026-05-16', 'entrepot': 'OMG-LIL'}]

--- referent NordDrive demandant FreshMarket : ramene a NordDrive ---
GET /commandes-clients (params={'client': 'FreshMarket', 'limit': 2}) -> 200
[{'id': 1, 'client': 'NordDrive', 'commande_id': 'ND-000549', 'date_commande': '2026-07-19', 'entrepot': 'OMG-LIL'}, {'id': 2, 'client': 'NordDrive', 'commande_id': 'ND-000220', 'date_commande': '2026-05-16', 'entrepot': 'OMG-LIL'}]


## 4. `/livraisons` : fermé aux référents clients (données de transport)

In [5]:
print("--- referent NordDrive sur /livraisons : 403 attendu ---")
_ = call("/livraisons", key=NORDDRIVE_KEY)

print()
print("--- Data Analyst sur /livraisons : autorise ---")
_ = call("/livraisons", key=ANALYST_KEY, statut="Livree", limit=2)


--- referent NordDrive sur /livraisons : 403 attendu ---
GET /livraisons (params={}) -> 403
{'detail': 'Réservé aux équipes internes (Data Engineers/Analysts)'}

--- Data Analyst sur /livraisons : autorise ---
GET /livraisons (params={'statut': 'Livree', 'limit': 2}) -> 200
[{'id': 1, 'tournee_id': 1, 'tracking_number': 'OMG0000001', 'heure_estimee': '15:15', 'heure_reelle': '16:45', 'statut': 'Livree'}, {'id': 3, 'tournee_id': 1, 'tracking_number': 'OMG0000095', 'heure_estimee': '11:45', 'heure_reelle': '13:30', 'statut': 'Livree'}]


## 5. Validation stricte du paramètre `statut`

`?statut=` est typé `Literal["Livree", "En cours"]` (voir `schemas.py`) :
une valeur hors de cet ensemble est rejetée (422), pas silencieusement
filtrée à vide.


In [6]:
_ = call("/livraisons", key=ANALYST_KEY, statut="en_retard")

GET /livraisons (params={'statut': 'en_retard'}) -> 422
{'detail': [{'type': 'literal_error', 'loc': ['query', 'statut'], 'msg': "Input should be 'Livree' or 'En cours'", 'input': 'en_retard', 'ctx': {'expected': "'Livree' or 'En cours'"}}]}


## 6. KPI taux de service, restreint au périmètre de l'appelant

In [7]:
print("--- tous clients (Data Analyst) ---")
_ = call("/kpis/taux-service", key=ANALYST_KEY)

print()
print("--- referent NordDrive : restreint a son client, meme sans le demander ---")
_ = call("/kpis/taux-service", key=NORDDRIVE_KEY)


--- tous clients (Data Analyst) ---
GET /kpis/taux-service (params={}) -> 200
[{'client': 'FreshMarket', 'nb_expeditions': 245, 'taux_service_pct': 90.6}, {'client': 'MedioTex', 'nb_expeditions': 259, 'taux_service_pct': 87.3}, {'client': 'NordDrive', 'nb_expeditions': 257, 'taux_service_pct': 87.5}]

--- referent NordDrive : restreint a son client, meme sans le demander ---
GET /kpis/taux-service (params={}) -> 200
[{'client': 'NordDrive', 'nb_expeditions': 257, 'taux_service_pct': 87.5}]


## Conclusion

Ce notebook confirme, contre une instance réelle de l'API, ce que
`tests/integration/test_api_routes.py` vérifie déjà avec des doubles :
authentification obligatoire, autorisation par rôle correctement
appliquée (restriction silencieuse du périmètre client, refus d'accès
aux livraisons pour les référents), et validation stricte des paramètres
de requête (`statut`).
